# 00 - Overview

Read the manifest, list the target GSEs and the project paths, and check which downloaded files are present.

**No AnnData loading happens here.** Run the download scripts first:

```bash
python scripts/00_validate_manifest.py
python scripts/01_download_geo_supplement.py
python scripts/02_extract_archives.py
python scripts/03_list_downloaded_files.py
```

In [ ]:
import sys
from pathlib import Path

# locate the v2 project root (folder containing config/dataset_manifest.yaml)
ROOT = Path.cwd()
while not (ROOT / "config" / "dataset_manifest.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
print("project root:", ROOT)

import manifest_utils as mf
man = mf.load_manifest()
paths = mf.project_paths(ROOT)
mf.ensure_dirs(paths)

In [ ]:
import pandas as pd

rows = []
for ds in mf.list_datasets(man):
    rows.append({
        'dataset_id': ds['dataset_id'],
        'source_accession': ds['source_accession'],
        'parent_gse': ds.get('parent_gse'),
        'loader_hint': ds['loader_hint'],
        'data_status': ds.get('data_status'),
        'n_files': len(ds.get('files', [])),
        'output': ds['output'],
    })
overview = pd.DataFrame(rows)
overview

## Project paths

In [ ]:
for k, v in paths.items():
    print(f'{k:20s} {v}')

## File presence check (raw / extracted)

In [ ]:
from archive_utils import find_files

rows = []
for ds in mf.list_datasets(man):
    raw_dir = mf.dataset_raw_dir(paths, ds)
    ext_dir = mf.dataset_extracted_dir(paths, ds)
    raw_present = [f['name'] for f in ds['files'] if (raw_dir / f['name']).exists()]
    rows.append({
        'dataset_id': ds['dataset_id'],
        'raw_present': f"{len(raw_present)}/{len(ds['files'])}",
        'extracted_files': len(find_files(ext_dir)),
    })
pd.DataFrame(rows)

Next: **01_load_each_gse_to_anndata.ipynb**